# Data provenance and validation

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
assert (REPORTS / "run_manifest.json").exists(), "Run python -m fraudgraph.pipeline --download first"
manifest = json.loads((REPORTS / "run_manifest.json").read_text())
print("Evidence run:", manifest["run_id"])
def figure(name):
    display(Image(filename=str(REPORTS / "figures" / name)))


Evidence run: 20260923T163139886696Z


The unit is a Bitcoin transaction. Illicit=1, licit=2, unknown=3. Unknown labels are missing outcomes, not negative examples. These notebooks inspect the full executed pipeline, whose logic lives in `src/fraudgraph`; they do not silently retrain models.

In [2]:
validation = json.loads((REPORTS / "data_validation.json").read_text())
display(pd.Series({k:v for k,v in validation.items() if k not in ["files", "timesteps", "missing_by_column"]}))
display(pd.DataFrame(validation["files"]).T)

nodes                                                                     203769
raw_edges                                                                 234355
unique_edges                                                              234355
duplicate_ids                                                                  0
duplicate_edges                                                                0
self_loops                                                                     0
missing_feature_cells                                                      16405
class_counts                                {'1': 4545, '2': 42019, '3': 157205}
same_step_edges                                                           234355
forward_step_edges                                                             0
backward_step_edges                                                            0
duplicate_local_feature_rows                                                2529
source_repository           

,sha256,bytes
txs_features.csv,2db326ec8ddb68f1d810c1834e1ff62e0a8300378f0984...,694789588
txs_classes.csv,013a11742969071a906878ded0319571df0657f9b7133e...,2361914
txs_edgelist.csv,a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02df...,4470584


In [3]:
f = pd.read_parquet(ROOT / "data/processed/transactions.parquet")
assert f.txId.is_unique
assert f.loc[f["class"] == 3, "y"].isna().all()
display(pd.crosstab(f["split"], f["class"]))
display(pd.read_csv(REPORTS / "missingness_by_split_class.csv"))

class,1,2,3
split,,,
test,636,10548,35463
train,2871,23510,94423
validation,1038,7961,27319


,split,class,transactions,missing_named
0,test,1,636,0
1,test,2,10548,110
2,test,3,35463,194
3,train,1,2871,0
4,train,2,23510,154
5,train,3,94423,133
6,validation,1,1038,0
7,validation,2,7961,255
8,validation,3,27319,119


The 965 rows missing added transaction attributes are retained. Imputation fits only on known training outcomes. The raw source's SHA-256 hashes identify the exact data; dataset files are excluded from Git.